<a href="https://colab.research.google.com/github/ymathurcode123/Yash-Mathur---2026B15410103/blob/main/Yash_Mathur_2026B15410103_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 15: Movie Theater Box Office & Concession Analysis

**Name:** Yash Mathur  
**Admission No:** 2026B15410103  
**Industry:** Entertainment / Media  

Data handling only with pandas and NumPy (no data visualization, no matplotlib).  
**How to run in Google Colab:** run all cells (Runtime > Run all). When asked, upload `15_cinema_sales.csv`. At the end, all results download as ONE Excel file: `cinema_analysis_results.xlsx`.

## Setup and helper functions

In [ ]:
# Project 15: Movie Theater Box Office & Concession Analysis

import io
from pathlib import Path
import numpy as np
import pandas as pd

# --- Dataset ----------------------------------------------------------------
# Put 15_cinema_sales.csv in the Colab Files panel (left side) before running.
# If it is missing or has the wrong columns, the standard project dataset is created automatically.
DATA = Path("15_cinema_sales.csv")

OUT = Path("outputs")
OUT.mkdir(exist_ok=True)
XLSX = OUT / "cinema_analysis_results.xlsx"
sheets = {}   # sheet name -> DataFrame, written to ONE Excel workbook at the end

report_buffer = io.StringIO()


def show_section(title, obj=None):
    """Print a titled section to the screen and store it for the Full_Report sheet."""
    text = f"\n{'=' * 70}\n{title}\n{'=' * 70}\n"
    if obj is not None:
        text += obj.to_string() if hasattr(obj, "to_string") else str(obj)
        text += "\n"
    print(text, end="")
    report_buffer.write(text)


def save(df, name):
    """Collect a result table; it becomes one sheet in the Excel workbook."""
    if not isinstance(df.index, pd.RangeIndex) or df.index.name:
        df = df.reset_index().rename(columns={"index": "Item"})
    sheets[name] = df


pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)


def make_dataset():
    """Create the standard 1000-record cinema dataset (same data as the provided CSV, seed 42)."""
    rng = np.random.default_rng(42)
    N = 1000

    movies = {
        "Galaxy Wars": "Sci-Fi", "Midnight Heist": "Thriller", "Love in Paris": "Romance",
        "Laugh Riot": "Comedy", "The Last Kingdom": "Action", "Haunted Manor": "Horror",
        "Toon Town Adventure": "Animation", "Deep Blue Secrets": "Documentary",
        "Steel Fury": "Action", "Dream Weavers": "Fantasy", "Cosmic Drift": "Sci-Fi",
        "Wedding Crashers 3": "Comedy", "Shadow Protocol": "Thriller", "Hearts Aflame": "Romance",
        "Night of Screams": "Horror", "Jungle Pals": "Animation",
    }
    titles = list(movies)
    popularity = rng.dirichlet(np.ones(len(titles)) * 2)

    formats = ["Standard 2D", "3D", "IMAX"]
    fmt_p = [0.6, 0.25, 0.15]
    fmt_price = {"Standard 2D": 180, "3D": 250, "IMAX": 350}
    fmt_occ = {"Standard 2D": 58, "3D": 64, "IMAX": 74}

    slots = ["Matinee", "Evening", "Late Night"]
    slot_p = [0.25, 0.5, 0.25]
    slot_occ = {"Matinee": -6, "Evening": 8, "Late Night": -2}

    dates = pd.date_range("2025-01-01", "2025-12-31")
    show_date = rng.choice(dates, N)

    movie = rng.choice(titles, N, p=popularity)
    fmt = rng.choice(formats, N, p=fmt_p)
    slot = rng.choice(slots, N, p=slot_p)

    df = pd.DataFrame({
        "Transaction_ID": [f"TXN{100000 + i}" for i in range(N)],
        "Show_Date": pd.to_datetime(show_date).strftime("%d-%m-%Y"),
        "Movie_Title": movie,
        "Genre": [movies[m] for m in movie],
        "Screen_Format": fmt,
        "Showtime_Slot": slot,
    })

    is_weekend = pd.to_datetime(show_date).dayofweek >= 5
    tickets = rng.choice([1, 2, 3, 4, 5, 6, 8, 10, 12], N,
                         p=[0.12, 0.34, 0.14, 0.17, 0.08, 0.06, 0.04, 0.03, 0.02])
    tickets = np.where(is_weekend & (rng.random(N) < 0.2), tickets + 2, tickets)
    df["Tickets_Sold"] = tickets

    price = df["Screen_Format"].map(fmt_price) + rng.choice([-20, 0, 20], N)
    df["Ticket_Price"] = price

    occ = (df["Screen_Format"].map(fmt_occ) + df["Showtime_Slot"].map(slot_occ)
           + np.where(is_weekend, 8, 0) + rng.normal(0, 7, N))
    df["Occupancy_Pct"] = np.clip(occ, 15, 100).round(1)

    has_snack = rng.random(N) < 0.72
    snack = (tickets * rng.uniform(60, 190, N)).round(0)
    df["Concession_Spend"] = np.where(has_snack, snack, 0.0)

    # ---- inject data-quality problems -------------------------------------
    miss_idx = rng.choice(N, 55, replace=False)
    df.loc[miss_idx, "Concession_Spend"] = np.nan

    dup_src = rng.choice(np.arange(20, N), 18, replace=False)   # 18 duplicate IDs
    dup_dst = rng.choice(np.arange(0, 20), 18, replace=False)
    for s, d in zip(dup_src, dup_dst):
        df.loc[s, "Transaction_ID"] = df.loc[d, "Transaction_ID"]
    return df


## Task 1

In [ ]:
EXPECTED = ["Transaction_ID", "Show_Date", "Movie_Title", "Genre", "Screen_Format",
            "Showtime_Slot", "Tickets_Sold", "Ticket_Price", "Occupancy_Pct", "Concession_Spend"]
key = lambda name: "".join(ch for ch in str(name).lower() if ch.isalnum())
lookup = {key(c): c for c in EXPECTED}


def standardise(d):
    """Make column names match exactly (ignores spaces, capitals, hidden characters)."""
    d.columns = [lookup.get(key(c), str(c).strip()) for c in d.columns]
    if "Show_Date" not in d.columns:                    # e.g. a column simply called "Date"
        date_like = [c for c in d.columns if "date" in c.lower()]
        if date_like:
            d = d.rename(columns={date_like[0]: "Show_Date"})
    return d


df = standardise(pd.read_csv(DATA)) if DATA.exists() else None
if df is None or not set(EXPECTED) <= set(df.columns):
    found = None if df is None else list(df.columns)
    print("NOTE: 15_cinema_sales.csv was not found or does not have all required columns.")
    print("      Columns found:", found)
    print("      Creating the standard project dataset instead (saved as 15_cinema_sales_generated.csv).")
    df = make_dataset()
    df.to_csv("15_cinema_sales_generated.csv", index=False)
show_section("TASK 1: Load & inspect dataset")
show_section("Shape (rows, columns)", df.shape)
show_section("First 5 rows", df.head())
buf = io.StringIO()
df.info(buf=buf)
show_section("Info", buf.getvalue())
show_section("Describe (numeric)", df.describe().round(2))

NOTE: 15_cinema_sales.csv was not found or does not have all required columns.
      Columns found: ['Transaction_ID', 'Movie_Title', 'Genre', 'Screen_Format', 'Showtime_Slot', 'Tickets_Sold', 'Ticket_Price', 'Occupancy_Pct', 'Concession_Spend']
      Creating the standard project dataset instead (saved as 15_cinema_sales_generated.csv).

TASK 1: Load & inspect dataset

Shape (rows, columns)
(1000, 10)

First 5 rows
  Transaction_ID   Show_Date    Movie_Title      Genre Screen_Format Showtime_Slot  Tickets_Sold  Ticket_Price  Occupancy_Pct  Concession_Spend
0      TXN100000  18-06-2025     Steel Fury     Action            3D    Late Night             2           250           60.5             259.0
1      TXN100001  17-02-2025  Love in Paris    Romance            3D       Evening             3           230           76.0             269.0
2      TXN100002  08-09-2025  Love in Paris    Romance            3D       Matinee             2           230           69.7             237.0
3   

## Task 2

In [ ]:
show_section("TASK 2: Missing values & duplicate Transaction_ID")
missing = df.isnull().sum().to_frame("Missing_Count")
missing["Missing_%"] = (missing["Missing_Count"] / len(df) * 100).round(2)
show_section("Missing values per column", missing)
dup_mask = df.duplicated(subset="Transaction_ID", keep=False)
dups = df[dup_mask].sort_values("Transaction_ID")
show_section(f"Rows sharing a Transaction_ID: {len(dups)} "
    f"({df.duplicated('Transaction_ID').sum()} extra copies)", dups.head(10))
save(missing, "Missing_Values")
save(dups.reset_index(drop=True), "Duplicate_Records")

# Keep the first record of every Transaction_ID
df = df.drop_duplicates(subset="Transaction_ID", keep="first").reset_index(drop=True)
show_section("Rows after removing duplicate Transaction_IDs", len(df))


TASK 2: Missing values & duplicate Transaction_ID

Missing values per column
                  Missing_Count  Missing_%
Transaction_ID                0        0.0
Show_Date                     0        0.0
Movie_Title                   0        0.0
Genre                         0        0.0
Screen_Format                 0        0.0
Showtime_Slot                 0        0.0
Tickets_Sold                  0        0.0
Ticket_Price                  0        0.0
Occupancy_Pct                 0        0.0
Concession_Spend             55        5.5

Rows sharing a Transaction_ID: 36 (18 extra copies)
    Transaction_ID   Show_Date          Movie_Title        Genre Screen_Format Showtime_Slot  Tickets_Sold  Ticket_Price  Occupancy_Pct  Concession_Spend
0        TXN100000  18-06-2025           Steel Fury       Action            3D    Late Night             2           250           60.5             259.0
223      TXN100000  10-07-2025    Deep Blue Secrets  Documentary          IMAX       Eve

## Task 3

In [ ]:
show_section("TASK 3: Clean missing Concession_Spend")
before = df["Concession_Spend"].isnull().sum()
# Justified strategy: spend depends on group size & format, so a global mean
# would be misleading. Use the MEDIAN of the same Screen_Format among buyers
# (spend > 0); median is robust against big group bookings.
buyers_median = (df[df["Concession_Spend"] > 0]
                 .groupby("Screen_Format")["Concession_Spend"].median())
df["Concession_Spend"] = df["Concession_Spend"].fillna(
    df["Screen_Format"].map(buyers_median))
show_section("Median spend of buyers per format (fill values)", buyers_median)
show_section(f"Missing Concession_Spend before: {before} | after", int(df["Concession_Spend"].isnull().sum()))


TASK 3: Clean missing Concession_Spend

Median spend of buyers per format (fill values)
Screen_Format
3D             333.0
IMAX           334.0
Standard 2D    344.0

Missing Concession_Spend before: 55 | after
0


## Task 4

In [ ]:
show_section("TASK 4: Datetime conversion & weekend flag")
df["Show_Date"] = pd.to_datetime(df["Show_Date"], dayfirst=True)
df["Weekday"] = df["Show_Date"].dt.day_name()
df["Is_Weekend"] = np.where(df["Weekday"].isin(["Saturday", "Sunday"]), "Weekend", "Weekday")
show_section("Sample", df[["Show_Date", "Weekday", "Is_Weekend"]].head())
show_section("Date range", f"{df['Show_Date'].min().date()} to {df['Show_Date'].max().date()}")


TASK 4: Datetime conversion & weekend flag

Sample
   Show_Date    Weekday Is_Weekend
0 2025-06-18  Wednesday    Weekday
1 2025-02-17     Monday    Weekday
2 2025-09-08     Monday    Weekday
3 2025-06-23     Monday    Weekday
4 2025-05-01   Thursday    Weekday

Date range
2025-01-02 to 2025-12-31


## Task 5

In [ ]:
show_section("TASK 5: Revenue calculations")
df["Ticket_Revenue"] = df["Tickets_Sold"] * df["Ticket_Price"]
df["Total_Spend"] = df["Ticket_Revenue"] + df["Concession_Spend"]
revenue = pd.Series({
    "Total Ticket Revenue": df["Ticket_Revenue"].sum(),
    "Total Concession Revenue": df["Concession_Spend"].sum(),
    "Combined Revenue": df["Total_Spend"].sum(),
}, name="Amount (Rs)")
show_section("Revenue summary", revenue.round(2))
save(revenue.to_frame(), "Revenue_Summary")


TASK 5: Revenue calculations

Revenue summary
Total Ticket Revenue         778020.0
Total Concession Revenue     313159.0
Combined Revenue            1091179.0


## Task 6

In [ ]:
show_section("TASK 6: Top 10 movies")
movie_stats = df.groupby("Movie_Title").agg(
    Tickets_Sold=("Tickets_Sold", "sum"),
    Ticket_Revenue=("Ticket_Revenue", "sum"))
top_tickets = movie_stats.sort_values("Tickets_Sold", ascending=False).head(10)
top_revenue = movie_stats.sort_values("Ticket_Revenue", ascending=False).head(10)
show_section("Top 10 by total tickets sold", top_tickets)
show_section("Top 10 by ticket revenue", top_revenue)
save(top_tickets, "Top10_By_Tickets")
save(top_revenue, "Top10_By_Revenue")


TASK 6: Top 10 movies

Top 10 by total tickets sold
                     Tickets_Sold  Ticket_Revenue
Movie_Title                                      
Cosmic Drift                  348           83020
Jungle Pals                   290           64790
The Last Kingdom              289           63910
Midnight Heist                275           62340
Toon Town Adventure           275           61240
Steel Fury                    272           56710
Hearts Aflame                 235           54290
Galaxy Wars                   230           53040
Shadow Protocol               217           47550
Deep Blue Secrets             215           45260

Top 10 by ticket revenue
                     Tickets_Sold  Ticket_Revenue
Movie_Title                                      
Cosmic Drift                  348           83020
Jungle Pals                   290           64790
The Last Kingdom              289           63910
Midnight Heist                275           62340
Toon Town Adventure  

## Task 7

In [ ]:
show_section("TASK 7: Average occupancy by screen format")
occ_fmt = (df.groupby("Screen_Format")["Occupancy_Pct"].mean()
           .round(2).sort_values(ascending=False).to_frame("Avg_Occupancy_%"))
show_section("Average occupancy", occ_fmt)
save(occ_fmt, "Occupancy_By_Format")


TASK 7: Average occupancy by screen format

Average occupancy
               Avg_Occupancy_%
Screen_Format                 
IMAX                     78.05
3D                       68.23
Standard 2D              62.13


## Task 8

In [ ]:
show_section("TASK 8: Weekday vs weekend")
wk = df.groupby("Is_Weekend").agg(
    Transactions=("Transaction_ID", "count"),
    Tickets_Sold=("Tickets_Sold", "sum"),
    Ticket_Revenue=("Ticket_Revenue", "sum"),
    Avg_Ticket_Revenue=("Ticket_Revenue", "mean"),
    Avg_Concession_Spend=("Concession_Spend", "mean"),
    Total_Concession=("Concession_Spend", "sum")).round(2)
show_section("Weekday vs Weekend", wk)
save(wk, "Weekday_vs_Weekend")


TASK 8: Weekday vs weekend

Weekday vs Weekend
            Transactions  Tickets_Sold  Ticket_Revenue  Avg_Ticket_Revenue  Avg_Concession_Spend  Total_Concession
Is_Weekend                                                                                                        
Weekday              714          2477          552510              773.82                311.67          222534.0
Weekend              268          1009          225510              841.46                338.15           90625.0


## Task 9

In [ ]:
show_section("TASK 9: Attendance by showtime slot")
slot = df.groupby("Showtime_Slot").agg(
    Transactions=("Transaction_ID", "count"),
    Total_Attendance=("Tickets_Sold", "sum"),
    Avg_Tickets_Per_Booking=("Tickets_Sold", "mean"),
    Avg_Occupancy_Pct=("Occupancy_Pct", "mean")).round(2)
slot = slot.reindex(["Matinee", "Evening", "Late Night"])
show_section("Showtime slot summary", slot)
save(slot, "Attendance_By_Slot")


TASK 9: Attendance by showtime slot

Showtime slot summary
               Transactions  Total_Attendance  Avg_Tickets_Per_Booking  Avg_Occupancy_Pct
Showtime_Slot                                                                            
Matinee                 256               936                     3.66              58.97
Evening                 485              1762                     3.63              72.08
Late Night              241               788                     3.27              61.38


## Task 10

In [ ]:
show_section("TASK 10: Concession attach rate")
attach_rate = (df["Concession_Spend"] > 0).mean() * 100
show_section("Overall attach rate (%)", round(attach_rate, 2))
attach_by_fmt = ((df["Concession_Spend"] > 0).groupby(df["Screen_Format"]).mean() * 100
                 ).round(2).to_frame("Attach_Rate_%")
attach_by_slot = ((df["Concession_Spend"] > 0).groupby(df["Showtime_Slot"]).mean() * 100
                  ).round(2).to_frame("Attach_Rate_%")
show_section("Attach rate by screen format", attach_by_fmt)
show_section("Attach rate by showtime slot", attach_by_slot)
attach_tbl = pd.concat([
    pd.DataFrame({"Group": "Overall", "Category": "All", "Attach_Rate_%": [round(attach_rate, 2)]}),
    attach_by_fmt.reset_index().rename(columns={"Screen_Format": "Category"}).assign(Group="Screen Format"),
    attach_by_slot.reset_index().rename(columns={"Showtime_Slot": "Category"}).assign(Group="Showtime Slot"),
], ignore_index=True)[["Group", "Category", "Attach_Rate_%"]]
save(attach_tbl, "Concession_Attach_Rate")


TASK 10: Concession attach rate

Overall attach rate (%)
73.42

Attach rate by screen format
               Attach_Rate_%
Screen_Format               
3D                     72.84
IMAX                   75.00
Standard 2D            73.24

Attach rate by showtime slot
               Attach_Rate_%
Showtime_Slot               
Evening                74.02
Late Night             71.37
Matinee                74.22


## Task 11

In [ ]:
show_section("TASK 11: Avg ticket & snack spending by genre")
genre = df.groupby("Genre").agg(
    Avg_Ticket_Revenue=("Ticket_Revenue", "mean"),
    Avg_Concession_Spend=("Concession_Spend", "mean")).round(2)
genre = genre.sort_values("Avg_Concession_Spend", ascending=False)
show_section("Genre comparison", genre)
save(genre, "Spend_By_Genre")


TASK 11: Avg ticket & snack spending by genre

Genre comparison
             Avg_Ticket_Revenue  Avg_Concession_Spend
Genre                                                
Fantasy                  817.44                390.21
Horror                   810.82                345.90
Animation                829.14                344.97
Sci-Fi                   938.34                336.53
Comedy                   685.50                313.96
Thriller                 784.93                308.78
Documentary              685.76                303.17
Action                   763.42                296.15
Romance                  730.62                281.49


## Task 12

In [ ]:
show_section("TASK 12: NumPy statistics on total transaction spend")
spend = df["Total_Spend"].to_numpy()
stats = pd.Series({"Mean": np.mean(spend), "Median": np.median(spend),
                   "Std Dev": np.std(spend, ddof=1)}, name="Total_Spend")
show_section("Statistics", stats.round(2))
save(stats.round(2).to_frame(), "Total_Spend_Statistics")


TASK 12: NumPy statistics on total transaction spend

Statistics
Mean       1111.18
Median      900.00
Std Dev     807.70


## Task 13

In [ ]:
show_section("TASK 13: Outliers in Tickets_Sold (IQR)")
q1, q3 = np.percentile(df["Tickets_Sold"], [25, 75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["Tickets_Sold"] < low) | (df["Tickets_Sold"] > high)]
show_section("IQR bounds", f"Q1={q1}, Q3={q3}, IQR={iqr}, lower={low}, upper={high}")
show_section(f"Large group bookings flagged: {len(outliers)}",
    outliers[["Transaction_ID", "Movie_Title", "Tickets_Sold"]].head(10))
save(outliers.reset_index(drop=True), "Group_Booking_Outliers")


TASK 13: Outliers in Tickets_Sold (IQR)

IQR bounds
Q1=2.0, Q3=4.0, IQR=2.0, lower=-1.0, upper=7.0

Large group bookings flagged: 81
    Transaction_ID        Movie_Title  Tickets_Sold
11       TXN100011     Midnight Heist            12
21       TXN100021  Deep Blue Secrets             8
28       TXN100028       Cosmic Drift            10
56       TXN100059   The Last Kingdom            12
58       TXN100062        Jungle Pals            10
64       TXN100068        Jungle Pals             8
79       TXN100083        Galaxy Wars            10
91       TXN100095    Shadow Protocol            10
145      TXN100151        Jungle Pals            10
170      TXN100176    Shadow Protocol            12


## Task 14

In [ ]:
show_section("TASK 14: Ticket-quantity bands")
df["Group_Size"] = pd.cut(df["Tickets_Sold"], bins=[0, 1, 2, 5, np.inf],
                          labels=["Solo", "Couple", "Small Group", "Large Group"])
bands = df.groupby("Group_Size", observed=True).agg(
    Bookings=("Transaction_ID", "count"),
    Avg_Concession_Spend=("Concession_Spend", "mean"),
    Total_Revenue=("Total_Spend", "sum")).round(2)
bands["Share_%"] = (bands["Bookings"] / bands["Bookings"].sum() * 100).round(2)
show_section("Group-size summary (Solo=1, Couple=2, Small=3-5, Large=6+)", bands)
save(bands, "Group_Size_Bands")


TASK 14: Ticket-quantity bands

Group-size summary (Solo=1, Couple=2, Small=3-5, Large=6+)
             Bookings  Avg_Concession_Spend  Total_Revenue  Share_%
Group_Size                                                         
Solo              102                102.76        33602.0    10.39
Couple            327                182.94       204780.0    33.30
Small Group       401                360.36       486883.0    40.84
Large Group       152                647.07       365914.0    15.48


## Task 15

In [ ]:
show_section("TASK 15: Marketing & scheduling recommendations (data driven)")
best_slot = slot["Avg_Occupancy_Pct"].idxmax()
weak_slot = slot["Avg_Occupancy_Pct"].idxmin()
best_fmt = occ_fmt.index[0]
top_genre_snack = genre.index[0]
wk_lift = (wk.loc["Weekend", "Avg_Ticket_Revenue"] / wk.loc["Weekday", "Avg_Ticket_Revenue"] - 1) * 100
solo_snack = bands.loc["Solo", "Avg_Concession_Spend"]
large_snack = bands.loc["Large Group", "Avg_Concession_Spend"]
recs = [
    f"1. Add more screens/shows in the '{best_slot}' slot - it has the highest average occupancy "
    f"({slot.loc[best_slot, 'Avg_Occupancy_Pct']}%).",
    f"2. Run discounted combo offers in the '{weak_slot}' slot "
    f"({slot.loc[weak_slot, 'Avg_Occupancy_Pct']}% occupancy) to lift attendance.",
    f"3. Prioritise {best_fmt} screens for blockbuster titles - highest occupancy "
    f"({occ_fmt.iloc[0, 0]}%).",
    f"4. Weekend bookings earn {wk_lift:.1f}% more ticket revenue per transaction - "
    f"launch weekend-only family combos and advance-booking promos.",
    f"5. '{top_genre_snack}' audiences spend the most on snacks; push premium combos on those "
    f"shows. Large groups spend Rs {large_snack:.0f} vs Rs {solo_snack:.0f} for solo viewers - "
    f"bundle group/party packs. Overall attach rate is {attach_rate:.1f}%, so add counter-side "
    f"upsell for the remaining buyers.",
]
for r in recs:
    show_section(r)
sheets["Recommendations"] = pd.DataFrame({"Recommendation": recs})


TASK 15: Marketing & scheduling recommendations (data driven)

1. Add more screens/shows in the 'Evening' slot - it has the highest average occupancy (72.08%).

2. Run discounted combo offers in the 'Matinee' slot (58.97% occupancy) to lift attendance.

3. Prioritise IMAX screens for blockbuster titles - highest occupancy (78.05%).

4. Weekend bookings earn 8.7% more ticket revenue per transaction - launch weekend-only family combos and advance-booking promos.

5. 'Fantasy' audiences spend the most on snacks; push premium combos on those shows. Large groups spend Rs 647 vs Rs 103 for solo viewers - bundle group/party packs. Overall attach rate is 73.4%, so add counter-side upsell for the remaining buyers.


## Save all results into one Excel workbook

In [ ]:
sheets["Cleaned_Data"] = df
sheets["Full_Report"] = pd.DataFrame({"Report": report_buffer.getvalue().splitlines()})


def write_workbook(path, tables):
    """Write every collected table into a single formatted .xlsx file."""
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
    order = ["Index"] + list(tables)
    index = pd.DataFrame({"Sheet": list(tables),
                          "Rows": [len(t) for t in tables.values()],
                          "Columns": [len(t.columns) for t in tables.values()]})
    tables = {"Index": index, **tables}
    with pd.ExcelWriter(path, engine="openpyxl", datetime_format="DD-MM-YYYY") as xw:
        for name in order:
            t = tables[name]
            t.to_excel(xw, sheet_name=name[:31], index=False)
            ws = xw.sheets[name[:31]]
            for c in ws[1]:
                c.font = Font(bold=True, color="FFFFFF")
                c.fill = PatternFill("solid", fgColor="1F4E78")
                c.alignment = Alignment(horizontal="center", vertical="center")
            ws.freeze_panes = "A2"
            for i, col in enumerate(t.columns, start=1):
                width = max([len(str(col))] + [len(str(v)) for v in t[col].head(200)]) + 2
                ws.column_dimensions[get_column_letter(i)].width = min(width, 60 if name != "Full_Report" else 140)
            if name == "Recommendations":
                ws.column_dimensions["A"].width = 130
                for r in ws.iter_rows(min_row=2):
                    r[0].alignment = Alignment(wrap_text=True, vertical="top")


write_workbook(XLSX, sheets)
print(f"\nDone. All results saved in ONE workbook: {XLSX}")

# Download the Excel file (Google Colab)
try:
    from google.colab import files
    files.download(str(XLSX))
except ImportError:
    pass


Done. All results saved in ONE workbook: outputs/cinema_analysis_results.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>